In [1]:
# 1. Check the GPU: all team runs use the same GPU type (L4)
import torch
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, f"{gpu.total_memory / 2**30:.1f} GiB")
assert "L4" in gpu.name, "Switch the runtime to an L4 GPU (Runtime > Change runtime type)"

NVIDIA L4 22.0 GiB


In [2]:
# 2. Get a fresh copy of main
%cd /content
!rm -rf kv-evidence-loss
!git clone -q https://github.com/burakzor/kv-evidence-loss.git
%cd /content/kv-evidence-loss
!git log -1 --format="%h %ad %s" --date=short

/content
/content/kv-evidence-loss
cb3d102 2026-11-22 Now icr follows convention.


In [3]:
# 3. Install the pinned team environment (torch is Colab's own CUDA build)
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|tokenizers|accelerate|datasets|pandas|numpy|pyarrow|tqdm) "

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
accelerate                            1.15.0
datasets                              4.8.5
numpy                                 2.4.6
pandas                                2.2.3
pyarrow                               23.0.1
tokenizers                            0.22.2
torch                                 2.14.1+cu130
tqdm                                  4.67.3
transformers                          5.2.0


In [4]:
# 4. Record the full environment of this run (GPU, driver, Python, CUDA, repository commit, every package); both budgets come from the same run
!mkdir -p results/answers
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"; echo "commit $(git rev-parse --short HEAD)"; pip freeze) > results/answers/qrhead_0.2_env.txt
!cp results/answers/qrhead_0.2_env.txt results/answers/qrhead_0.1_env.txt
!head -5 results/answers/qrhead_0.2_env.txt

NVIDIA L4, 580.82.07
Python 3.13.16
torch 2.14.1+cu130 cuda 13.0
commit cb3d102
absl-py==2.0.0


In [5]:
# 5. Optional: Hugging Face token for a faster download (only works in the Colab web UI)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"No HF token ({type(e).__name__}); downloading anonymously.")

No HF token (SecretNotFoundError); downloading anonymously.


In [6]:
# 6. First run on 3 questions: answers to /content/qrhead_test, masks to results/raw/kept_limit3 (the team's test-run folder), so that nothing mixes with the full run
# (first run downloads Qwen3-8B, ~16 GB)
!cd scripts && python -c "from pathlib import Path; import qrhead_run as q; q.run(q.MODEL_NAME, q.HEADS_PATH, 3, Path('/content/qrhead_test/answers'), q.ROOT / 'results' / 'raw' / 'kept_limit3' / 'qrhead')"

README.md: 100% 9.52k/9.52k [00:00<00:00, 3.77MB/s]

distractor/train-00000-of-00002.parquet: downloading bytes:  76% 126M/166M [00:02<00:00, 104MB/s, 10.0MB/s  ] 
distractor/train-00000-of-00002.parquet: downloading bytes:  95% 158M/166M [00:02<00:00, 126MB/s, 12.8MB/s  ]
distractor/train-00000-of-00002.parquet: reconstructing file:  42% 69.7M/166M [00:02<00:02, 32.7MB/s, 1.57MB/s  ]
distractor/train-00000-of-00002.parquet: downloading bytes: 100% 166M/166M [00:03<00:00, 53.4MB/s, 14.8MB/s  ]
distractor/train-00000-of-00002.parquet: reconstructing file: 100% 166M/166M [00:03<00:00, 53.4MB/s, 15.3MB/s  ]

distractor/train-00001-of-00002.parquet: downloading bytes:   0% 0.00/166M [00:00<?, ?B/s]
distractor/train-00001-of-00002.parquet: downloading bytes:  90% 150M/166M [00:02<00:00, 144MB/s, 12.1MB/s  ]
distractor/train-00001-of-00002.parquet: downloading bytes: 100% 166M/166M [00:02<00:00, 56.7MB/s, 14.9MB/s  ]
distractor/train-00001-of-00002.parquet: reconstructing file: 100% 166M/166

In [7]:
# 7. Full run: score the 10 paragraphs of all 169 clean questions with the QRHead heads, answer at budgets 0.2 and 0.1
!python scripts/qrhead_run.py

Loading weights: 100% 399/399 [00:04<00:00, 98.16it/s, Materializing param=model.norm.weight] 
100% 169/169 [04:31<00:00,  1.61s/it]

budget: 0.2 | questions: 169 | device: NVIDIA L4
contains_alias: 121 / 169 0.716
paragraphs kept: 2.4 | prompt kept fraction: 0.233
A kept: 0.935 | B kept: 0.805 | both kept: 0.763
rank_s: 0.683 | generate_s: 0.448 | peak_mem_gb: 15.91

budget: 0.1 | questions: 169 | device: NVIDIA L4
contains_alias: 86 / 169 0.509
paragraphs kept: 1.37 | prompt kept fraction: 0.14
A kept: 0.769 | B kept: 0.509 | both kept: 0.32
rank_s: 0.683 | generate_s: 0.465 | peak_mem_gb: 15.91

answers written to /content/kv-evidence-loss/results/answers | masks written to /content/kv-evidence-loss/results/raw/kept/qrhead


In [8]:
# 8. Look at the results per budget, next to the full-cache baseline
import pandas as pd
full_df = pd.read_parquet("results/answers/full_1.0.parquet")
summary_rows = []
for budget in [0.2, 0.1]:
    df = pd.read_parquet(f"results/answers/qrhead_{budget}.parquet")
    summary_rows.append({
        "budget": budget,
        "questions": len(df),
        "contains_alias": df["contains_alias"].mean(),
        "paragraphs_kept": df["n_paragraphs_kept"].mean(),
        "prompt_kept_fraction": df["prompt_kept_fraction"].mean(),
        "a_kept": df["a_kept"].mean(),
        "b_kept": df["b_kept"].mean(),
        "both_kept": (df["a_kept"] & df["b_kept"]).mean(),
        "rank_s": df["rank_s"].mean(),
        "generate_s": df["generate_s"].mean(),
        "peak_mem_gb": df["peak_mem_gb"].max(),
    })
print("full cache contains_alias:", round(full_df["contains_alias"].mean(), 3))
print(pd.DataFrame(summary_rows).round(3).to_string(index=False))
df = pd.read_parquet("results/answers/qrhead_0.1.parquet")
print(df["n_paragraphs_kept"].value_counts().sort_index())

full cache contains_alias: 0.775
 budget  questions  contains_alias  paragraphs_kept  prompt_kept_fraction  a_kept  b_kept  both_kept  rank_s  generate_s  peak_mem_gb
    0.2        169           0.716            2.402                 0.233   0.935   0.805      0.763   0.683       0.448       15.906
    0.1        169           0.509            1.373                 0.140   0.769   0.509      0.320   0.683       0.465       15.906
n_paragraphs_kept
1    108
2     59
3      2
Name: count, dtype: int64


In [ ]:
# 9. Zip the kept masks (paths inside the zip: qrhead/{budget}/{id}.npz, unzip into results/raw/kept/) and copy them to Google Drive (the Colab disk is deleted when the session ends)
!ls results/raw/kept/qrhead/0.2 | wc -l
!ls results/raw/kept/qrhead/0.1 | wc -l
!cd results/raw/kept && zip -qr /content/qrhead_masks.zip qrhead
!ls -lh /content/qrhead_masks.zip
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/kv-evidence-loss/raw/kept
!cp -r results/raw/kept/qrhead /content/drive/MyDrive/kv-evidence-loss/raw/kept/
!cp /content/qrhead_masks.zip /content/drive/MyDrive/kv-evidence-loss/raw/

169
169
-rw-r--r-- 1 root root 175K Nov 22 14:37 /content/qrhead_masks.zip


In [ ]:
# 10. Download the answers tables, the run settings, the environment records and the zipped masks
from google.colab import files
for budget in [0.2, 0.1]:
    files.download(f"results/answers/qrhead_{budget}.parquet")
    files.download(f"results/answers/qrhead_{budget}_config.json")
    files.download(f"results/answers/qrhead_{budget}_env.txt")
files.download("/content/qrhead_masks.zip")